In [1]:
# 7-6-2026

In [18]:
import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix
from scipy.stats import spearmanr, kendalltau

In [19]:
# load embeddings, set domain_id as index
df_embeddings = pd.read_csv("domain_embeddings.csv")
df_embeddings.set_index("domain_id", inplace=True)

In [20]:
df_embeddings.head()

,e_0,e_1,e_2,e_3,e_4,e_5,e_6,e_7
domain_id,,,,,,,,
0,0.117884,0.232503,0.171468,-0.283202,-0.131189,-0.261552,0.254024,0.005274
1,0.966123,0.417381,0.218418,-0.573192,0.454459,0.111833,1.051024,-0.564022
2,-0.273094,-0.179182,-0.160054,-0.191988,-0.161651,-0.020716,0.082488,0.359735
4,-0.376840,0.847378,0.296678,0.529982,-0.204373,-0.289536,-0.807122,-0.032328
5,0.165855,0.146730,0.199480,-0.341562,0.070639,-0.427978,0.268521,0.048948


In [21]:
# raw embedding distance matrix, no normalization
dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
# no scaling because the fact that some nodes have greater scales than others actually means something for downstream prediction
# nodes having differnt scales actually means something to the network

C:\Users\Yash\AppData\Local\Temp\ipykernel_32256\2764295889.py:2: DeprecationWarning: `distance_matrix` is deprecated in favor of `scipy.spatial.distance.cdist` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
C:\Users\Yash\AppData\Local\Temp\ipykernel_32256\2764295889.py:2: DeprecationWarning: `minkowski_distance` is deprecated in favor of `scipy.spatial.distance.minkowski` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
C:\Users\Yash\AppData\Local\Temp\ipykernel_32256\2764295889.py:2: DeprecationWarning: `minkowski_distance_p` is deprecated in favor of `scipy.spatial.distance.minkowski` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default


In [22]:
# negate so bigger=lower transferability. same  as rawdist
dist_raw_df = pd.DataFrame(-dist_raw, index=df_embeddings.index, columns=df_embeddings.index)

In [23]:
dist_raw_df.head()

domain_id,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
domain_id,,,,,,,,,,,,,,,,,,,,,
0,-0.000000,-1.510527,-0.809057,-1.559922,-0.290514,-0.791346,-0.851834,-0.578394,-0.870551,-1.631771,...,-0.749973,-0.833351,-0.788179,-1.226697,-0.563868,-2.669826,-1.404046,-2.563566,-0.645691,-0.724334
1,-1.510527,-0.000000,-2.090132,-2.746294,-1.481368,-2.106552,-1.961124,-1.052794,-1.979368,-2.275023,...,-1.823727,-2.080568,-2.139457,-1.290034,-1.835946,-3.041813,-2.113099,-3.627574,-1.996643,-1.730952
2,-0.809057,-2.090132,-0.000000,-1.677392,-0.895293,-0.949989,-0.782254,-1.164730,-1.232974,-1.532002,...,-0.965358,-0.943903,-0.742716,-1.816586,-0.507480,-2.555211,-1.457728,-2.788868,-0.506546,-0.787000
4,-1.559922,-2.746294,-1.677392,-0.000000,-1.677163,-0.804259,-1.719471,-1.981250,-1.606651,-2.439294,...,-1.672355,-1.252772,-1.160795,-2.419270,-1.583679,-3.117080,-2.370836,-1.601897,-1.247870,-1.431690
5,-0.290514,-1.481368,-0.895293,-1.677163,-0.000000,-0.935469,-1.056932,-0.651210,-0.699007,-1.722002,...,-0.954050,-1.052628,-0.976576,-1.021278,-0.752971,-2.879188,-1.292905,-2.660425,-0.802733,-0.891219


In [24]:
dist_raw_df.to_csv("embedding_matrix.csv")

In [39]:
embedding_matrix = pd.read_csv("embedding_matrix.csv")
embedding_matrix.set_index("domain_id", inplace=True)
embedding_matrix.index.name = None
embedding_matrix.index = embedding_matrix.index.astype(int)
embedding_matrix.columns = embedding_matrix.columns.astype(int)
embedding_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-1.510527,-0.809057,-1.559922,-0.290514,-0.791346,-0.851834,-0.578394,-0.870551,-1.631771,...,-0.749973,-0.833351,-0.788179,-1.226697,-0.563868,-2.669826,-1.404046,-2.563566,-0.645691,-0.724334
1,-1.510527,-0.000000,-2.090132,-2.746294,-1.481368,-2.106552,-1.961124,-1.052794,-1.979368,-2.275023,...,-1.823727,-2.080568,-2.139457,-1.290034,-1.835946,-3.041813,-2.113099,-3.627574,-1.996643,-1.730952
2,-0.809057,-2.090132,-0.000000,-1.677392,-0.895293,-0.949989,-0.782254,-1.164730,-1.232974,-1.532002,...,-0.965358,-0.943903,-0.742716,-1.816586,-0.507480,-2.555211,-1.457728,-2.788868,-0.506546,-0.787000
4,-1.559922,-2.746294,-1.677392,-0.000000,-1.677163,-0.804259,-1.719471,-1.981250,-1.606651,-2.439294,...,-1.672355,-1.252772,-1.160795,-2.419270,-1.583679,-3.117080,-2.370836,-1.601897,-1.247870,-1.431690
5,-0.290514,-1.481368,-0.895293,-1.677163,-0.000000,-0.935469,-1.056932,-0.651210,-0.699007,-1.722002,...,-0.954050,-1.052628,-0.976576,-1.021278,-0.752971,-2.879188,-1.292905,-2.660425,-0.802733,-0.891219
6,-0.791346,-2.106552,-0.949989,-0.804259,-0.935469,-0.000000,-1.050814,-1.241372,-1.090607,-1.880838,...,-1.035252,-0.703391,-0.535807,-1.818594,-0.820624,-2.736977,-1.789071,-2.083189,-0.505503,-0.724823
7,-0.851834,-1.961124,-0.782254,-1.719471,-1.056932,-1.050814,-0.000000,-1.034218,-1.536016,-1.478483,...,-0.644703,-0.618720,-0.584116,-1.887387,-0.456107,-2.112804,-1.693632,-2.552087,-0.819067,-0.926928
8,-0.578394,-1.052794,-1.164730,-1.981250,-0.651210,-1.241372,-1.034218,-0.000000,-1.298744,-1.817694,...,-1.051818,-1.192617,-1.193347,-1.151389,-0.839243,-2.670345,-1.697972,-2.955969,-1.099103,-0.912110
11,-0.870551,-1.979368,-1.232974,-1.606651,-0.699007,-1.090607,-1.536016,-1.298744,-0.000000,-2.100329,...,-1.331969,-1.324412,-1.269969,-1.100195,-1.251726,-3.394171,-1.319576,-2.425239,-1.088040,-1.389363
12,-1.631771,-2.275023,-1.532002,-2.439294,-1.722002,-1.880838,-1.478483,-1.817694,-2.100329,-0.000000,...,-1.219506,-1.722203,-1.658052,-2.436750,-1.602689,-1.705182,-1.278232,-3.219685,-1.572189,-1.623930


In [40]:
rawdist_matrix = pd.read_csv("../baselines/rawdist_matrix.csv")
rawdist_matrix.set_index("Unnamed: 0", inplace=True)
rawdist_matrix.index.name = None
rawdist_matrix.index = rawdist_matrix.index.astype(int)
rawdist_matrix.columns = rawdist_matrix.columns.astype(int)
rawdist_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-6.672423,-11.045133,-7.986460,-3.308284,-7.958821,-6.068376,-4.666705,-5.184888,-4.589527,...,-7.662378,-6.407206,-8.197987,-5.296501,-4.204545,-11.505974,-7.273316,-13.276326,-6.751253,-3.125536
1,-6.672423,-0.000000,-13.599319,-9.841238,-5.547623,-9.436429,-9.730264,-5.698760,-8.974480,-8.392720,...,-11.038897,-10.159206,-10.744922,-5.711799,-7.998925,-13.061897,-11.181847,-15.567050,-8.537365,-6.931799
2,-11.045133,-13.599319,-0.000000,-13.082436,-11.499412,-12.708249,-9.285989,-13.684210,-9.565427,-8.687225,...,-7.894825,-9.563264,-10.696453,-11.590126,-9.063832,-9.620238,-7.577379,-16.658596,-9.353973,-11.288658
4,-7.986460,-9.841238,-13.082436,-0.000000,-8.019986,-3.351193,-7.878595,-9.379059,-9.467587,-9.484902,...,-10.290864,-7.254905,-6.459036,-9.505884,-7.903673,-13.860378,-11.642435,-9.120748,-5.452087,-7.889157
5,-3.308284,-5.547623,-11.499412,-8.019986,-0.000000,-7.551036,-6.591061,-4.260023,-4.860621,-5.079144,...,-8.673302,-7.712744,-8.004912,-4.035172,-5.374218,-12.376469,-7.461498,-12.675088,-6.924344,-4.980700
6,-7.958821,-9.436429,-12.708249,-3.351193,-7.551036,-0.000000,-7.307292,-8.895403,-9.258151,-9.329919,...,-9.975772,-7.488843,-4.461359,-9.152761,-7.552179,-13.639737,-11.240952,-8.347249,-5.771723,-7.863910
7,-6.068376,-9.730264,-9.285989,-7.878595,-6.591061,-7.307292,-0.000000,-8.832793,-7.072502,-5.136909,...,-4.850997,-4.276811,-5.501850,-8.444666,-2.986468,-8.724586,-7.209533,-12.837734,-5.064828,-5.701705
8,-4.666705,-5.698760,-13.684210,-9.379059,-4.260023,-8.895403,-8.832793,-0.000000,-7.504786,-7.592999,...,-10.594288,-8.880334,-9.930977,-5.531485,-7.389167,-14.100157,-10.431665,-13.559093,-9.241698,-5.032279
11,-5.184888,-8.974480,-9.565427,-9.467587,-4.860621,-9.258151,-7.072502,-7.504786,-0.000000,-5.607953,...,-8.563762,-8.084519,-8.778158,-4.434500,-6.187654,-12.536701,-4.309561,-13.146039,-7.716434,-6.559332
12,-4.589527,-8.392720,-8.687225,-9.484902,-5.079144,-9.329919,-5.136909,-7.592999,-5.607953,-0.000000,...,-6.457259,-6.402053,-8.537121,-6.857717,-4.288140,-9.152598,-5.360677,-14.267618,-6.997054,-5.922538


In [41]:
true_matrix = pd.read_csv("../transfer-matrix/rf_transfer_matrix.csv")
true_matrix.set_index("Unnamed: 0", inplace=True)
true_matrix.index.name = None
true_matrix.index = true_matrix.index.astype(int)
true_matrix.columns = true_matrix.columns.astype(int)
true_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392727,0.291660,0.127238,0.058840,0.214817,0.084705,0.149056,0.182358,0.173877,0.026375,...,0.228505,0.243433,0.149609,0.275203,0.112525,-0.011557,0.116862,0.158796,0.025454,0.029928
1,0.142172,0.395760,0.030643,0.010376,0.180707,0.064432,0.056092,0.128808,0.097788,0.039590,...,0.167477,0.032806,0.029704,0.193033,0.055264,0.066550,0.108203,0.199762,0.023240,-0.037710
2,0.196600,0.214445,0.368173,0.023968,0.207323,0.140139,0.172990,0.231019,0.156741,0.057226,...,0.197402,0.223793,0.121457,0.248807,0.134485,0.053503,-0.005610,0.253965,0.031899,0.059566
4,0.235489,0.227173,0.127394,0.410382,0.215954,0.186207,0.125068,0.196603,0.165264,0.034514,...,0.178055,0.245668,0.147579,0.284360,0.217076,-0.003941,0.037643,0.306750,0.137154,0.020968
5,0.190803,0.206530,0.097887,0.087230,0.464604,0.144950,0.159361,0.204884,0.264641,0.104493,...,0.197195,0.232923,0.117228,0.292623,0.170687,-0.020694,0.115293,0.297271,0.067951,0.051770
6,0.178282,0.238235,0.034447,0.220869,0.191487,0.317381,0.118250,0.163631,0.119401,0.021430,...,0.202794,0.247076,0.125801,0.255555,0.201974,-0.068777,0.099243,0.321076,0.102840,-0.017236
7,0.113396,0.054282,0.061550,0.069370,0.044484,0.109329,0.329688,0.092023,0.159883,0.051387,...,0.184071,0.179713,0.142222,0.192045,0.209176,0.035736,0.042035,0.159817,0.008022,0.022467
8,0.181002,0.228547,-0.006209,0.023346,0.133774,0.117255,0.194411,0.424899,0.064289,-0.004919,...,0.093679,0.136592,0.101116,0.173933,0.095452,-0.003312,0.050942,0.240088,0.051530,0.007182
11,0.214508,0.227264,0.109767,0.124790,0.237549,0.076848,0.107019,0.207917,0.551275,-0.003236,...,0.256072,0.166287,0.102690,0.366209,0.181682,-0.007782,0.147501,0.206234,-0.057620,0.064859
12,0.118927,0.175802,-0.016922,0.112255,0.159920,0.070012,0.077276,0.026258,0.103184,0.502334,...,0.163311,-0.091433,0.060118,0.163172,0.119053,0.080449,0.089125,0.118189,0.068853,-0.070408


In [42]:
full_tau, _ = kendalltau(embedding_matrix.values.flatten(), true_matrix.values.flatten())
full_tau

np.float64(0.19555378571785756)

In [43]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = kendalltau(embedding_matrix.values[mask], true_matrix.values[mask])
print(f"off diag kendal tau: {off_diag_tau:.4f}")

off diag kendal tau: 0.1471


In [49]:
full_tau, _ = kendalltau(embedding_matrix.values.flatten(), rawdist_matrix.values.flatten())
full_tau

np.float64(0.49611384192489355)

In [45]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = kendalltau(embedding_matrix.values[mask], rawdist_matrix.values[mask])
print(f"off diag kendal tau: {off_diag_tau:.4f}")

off diag kendal tau: 0.4655


In [46]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = spearmanr(true_matrix.values[mask], embedding_matrix.values[mask])
print(f"off diag sparman: {off_diag_tau:.4f}")

off diag sparman: 0.2147


In [51]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = spearmanr(true_matrix.values[mask], rawdist_matrix.values[mask])
print(f"off diag sparman: {off_diag_tau:.4f}")

off diag sparman: 0.0943


In [52]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = spearmanr(rawdist_matrix.values[mask], true_matrix.values[mask])
print(f"off diag spearman: {off_diag_tau:.4f}")

off diag spearman: 0.0943
